# Dual-Axis GeoFormer — train on Colab's free GPU

This notebook does what `docs/MANUAL.md` §12 documents as the honest next step: the prototype's real-SpaceNet-8 experiment (`prepare_real_data.py`) worked end-to-end but only pulled 20 tiles on a CPU-only machine, which wasn't enough data to converge. Colab gives you a free GPU and enough time to pull a much larger real sample and actually train on it.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine).

**What this notebook does NOT do:** guarantee production accuracy. More real data and more epochs than the CPU prototype could manage, still likely well short of a full competitive SpaceNet-8 result — see the repo's `docs/MANUAL.md` for what "trained" should and shouldn't be claimed to mean at each stage.

In [ ]:
!git clone https://github.com/Redwan002117/D_A_GeoFormer.git
%cd D_A_GeoFormer
!pip install -q boto3 scikit-image

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go enable the GPU runtime!)')

## 1. Pull a much larger real SpaceNet-8 sample

The public `spacenet-dataset` S3 bucket needs no AWS credentials (unsigned access). The Germany AOI alone has 202 tiles; this pulls a larger, class-balanced-ish slice than the 20-tile CPU proof-of-concept. Raise `--n-tiles` further if you have the disk/time (each tile is a few MB).

In [ ]:
!python prepare_real_data.py --n-tiles 150 --out-dir real_sn8_dataset

## 2. Train from scratch on the real data

Starting fresh (no `--resume`) rather than continuing the CPU prototype's synthetic-converged checkpoint — with this much more real data and a GPU, a clean run is simpler to reason about than another resume. Swap in `--resume checkpoints/last.pt` (after uploading that checkpoint) if you'd rather continue from the synthetic-pretrained weights.

In [ ]:
!python train.py --data-dir real_sn8_dataset --image-size 256 --batch-size 16 \
                  --epochs 80 --lr 1e-3 --checkpoint-dir checkpoints \
                  --log-csv training_log_colab.csv

In [ ]:
!python plot_training_curve.py --log-csv training_log_colab.csv --out training_curve_colab.png
from IPython.display import Image, display
display(Image('training_curve_colab.png'))

## 3. Evaluate and visualize

Read the per-class F1 table honestly — background will always look easy; building/road/flooded are the numbers that actually matter for this task.

In [ ]:
!python evaluate.py --checkpoint checkpoints/best.pt --data-dir real_sn8_dataset --image-size 256

## 4. Download the trained checkpoint back to your machine

So you can run `demo.py` / `serve.py` locally against real-data-trained weights instead of the synthetic-only ones.

In [ ]:
from google.colab import files
files.download('checkpoints/best.pt')

## 5. Optional: scale to the full SpaceNet-8 benchmark

`prepare_real_data.py` currently only targets the `Germany_Training_Public` AOI (202 tiles). The full SN-8 release has more AOIs under the same bucket/prefix pattern (`spacenet/SN8_floods/<AOI>/`) — list them with the cell below, then extend `BASE` in `prepare_real_data.py` (or loop over AOIs) to pull the rest.

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.client import Config
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
resp = s3.list_objects_v2(Bucket='spacenet-dataset', Prefix='spacenet/SN8_floods/', Delimiter='/')
for p in resp.get('CommonPrefixes', []):
    print(p['Prefix'])